<a href="https://colab.research.google.com/github/Ayush-Singh-36/Transcribing_Model_PyTorch/blob/main/transcribing_training_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imported the GitHub Repository here

In [ ]:
import os
from google.colab import userdata

#retrieve GitHub PAT from colab secrets
github_pat = userdata.get('git_token')

if not github_pat:
  raise ValueError("GITHUB_PAT not found in colab secrets. please add it")

#Original repository URL
repository_url_base = "https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git"

# Constructed the authenticated URL
authenticated_repository_url = repository_url_base.replace("https://github.com", f"https://{github_pat}@github.com/")

print("Authenticated repository URL prepared for cloning.")

Authenticated repository URL prepared for cloning.


# Clone the repository here for model development at this session

In [ ]:
repository_url = "https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git"
!git clone {repository_url}
repository_name = repository_url.split("/")[-1].replace(".git", "")
print(f"Repository '{repository_name}' cloned successfully.")
#Using the authenticated URL for cloning
!git clone {authenticated_repository_url}
repository_name = authenticated_repository_url.split("/")[-1].replace(".git", "")
print(f"Repository '{repository_name}'cloned successfully.")

Cloning into 'Transcribing_Model_PyTorch'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 11 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 10.38 KiB | 10.38 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Repository 'Transcribing_Model_PyTorch' cloned successfully.
fatal: destination path 'Transcribing_Model_PyTorch' already exists and is not an empty directory.
Repository 'Transcribing_Model_PyTorch'cloned successfully.


# Download the Dataset directly from kaggle to our github directory imported in this notebook

In [ ]:
import os
from google.colab import userdata
import sys

def custom_exit(status):
  print(f"Kaggle API tried to exit with status {status}. Ignoring from colab evviornment.")
  #Optionally, raise an exception or log instead of actual exit
sys.exit = custom_exit
exit = custom_exit
try:
  __builtins__.exit = custom_exit

except AttributeError:
  print("Could not patch __builtins__.exit - it might not be present or modifiable in this enviornment.")


#retrieve credentials from colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

#now import and run the download code
from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "beomseongkim/torchaudio"
#update download_path to place data within the cloned repository
download_path = "./Transcribing_Model_PyTorch/data"

#create the directory if it doesn't exit
os.makedirs(download_path, exist_ok = True)
print("Authenticating via enviornment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading the dataset from kaggle...")
api.dataset_download_files(dataset_slug, path = download_path, unzip = True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")
#Change directory to the cloned repository
%cd /content/Transcribing_Model_PyTorch

Authenticating via enviornment variables...
Dataset URL: https://www.kaggle.com/datasets/beomseongkim/torchaudio
Done! Your files have been saved to the './Transcribing_Model_PyTorch/data' folder.
/content/Transcribing_Model_PyTorch


# Configure Git remote with Personal Access Token (PAT) for Authentication

In [ ]:
import os
from google.colab import userdata
#Retrieve GitHub PAT from google colab secrets
github_pat = userdata.get('git_token')
#We'll re-extract the base URL and repository name
repo_url_base = "https://github.com"
repo_path = repository_url[len(repo_url_base):]
#Construct the authenticated URL
authenticated_repo_url = f"https://{github_pat}@github.com/{repo_path}"
print("Authenticated repository URL created.")

#change to the cloned repository directory if not already there
%cd /content/Transcribing_Model_PyTorch

#set the remote origin to the authenticated URL
!git remote set-url origin (authenticated_repo_url)
#verify the remote URL has been updated
!git remote -v
print("Git remote origin configured with PAT for authentication.")

Authenticated repository URL created.
/content/Transcribing_Model_PyTorch
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `git remote set-url origin (authenticated_repo_url)'
origin	https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git (fetch)
origin	https://github.com/Ayush-Singh-36/Transcribing_Model_PyTorch.git (push)
Git remote origin configured with PAT for authentication.


# Making this dataset comfortable for execution in real-life
*As our Kaggle dataset is lossless audio we can't use it much for training the model for real-life application, that's why we need to noise to our kaggle dataset to make it actually useful*

In [ ]:
import torch
import torchaudio
import os
import random

# Define the base data path. Assuming current working directory is /content/Transcribing_Model_PyTorch
base_data_path = "/content/Transcribing_Model_PyTorch/data"
TRAIN_DIR = os.path.join(base_data_path, "train-clean-100/LibriSpeech/train-clean-100")
TEST_DIR = os.path.join(base_data_path, "test-clean/LibriSpeech/test-clean")

# Output directories for noisy data
NOISY_TRAIN_DIR = os.path.join(base_data_path, "train-noisy-100")
NOISY_TEST_DIR = os.path.join(base_data_path, "test-noisy")

# Check if TRAIN_DIR and TEST_DIR exist and have content
if not os.path.exists(TRAIN_DIR) or not os.listdir(TRAIN_DIR):
    print(f"Warning: TRAIN_DIR '{TRAIN_DIR}' is empty or does not exist. Please check your data download and directory structure.")
if not os.path.exists(TEST_DIR) or not os.listdir(TEST_DIR):
    print(f"Warning: TEST_DIR '{TEST_DIR}' is empty or does not exist. Please check your data download and directory structure.")

def add_gaussian_noise(waveform, sample_rate, snr_db=15):
    """
    Adds Gaussian noise to an audio waveform.

    Args:
        waveform (torch.Tensor): The input audio waveform. Shape (channels, samples).
        sample_rate (int): The sample rate of the audio (not directly used for noise generation, but good practice).
        snr_db (float): Desired Signal-to-Noise Ratio in dB.

    Returns:
        torch.Tensor: The waveform with added Gaussian noise.
    """
    # Ensure waveform is float
    if waveform.dtype != torch.float32:
        waveform = waveform.to(torch.float32)

    # Calculate signal power
    # Mean squared value is a common measure for signal power for zero-mean signals
    signal_power = torch.mean(waveform**2)

    if signal_power == 0:
        print("Warning: Signal power is zero. Cannot add noise with specified SNR. Returning original waveform.")
        return waveform

    # Convert SNR from dB to linear scale
    snr_linear = 10**(snr_db / 10)

    # Calculate noise power
    noise_power = signal_power / snr_linear

    # Generate Gaussian noise with calculated power
    # The standard deviation of the noise is sqrt(noise_power)
    noise = torch.randn_like(waveform) * torch.sqrt(noise_power)

    return waveform + noise

def process_audio_directory(input_root_dir, output_root_dir, snr_db=15):
    """
    Processes all audio files in a directory, adds noise, and saves them to an output directory,
    maintaining the original directory structure.

    Args:
        input_root_dir (str): The root directory containing original audio files.
        output_root_dir (str): The root directory where noisy audio files will be saved.
        snr_db (float): Desired Signal-to-Noise Ratio in dB for noise addition.
    """
    os.makedirs(output_root_dir, exist_ok=True)
    print(f"Processing audio files from '{input_root_dir}' to '{output_root_dir}' with SNR {snr_db}dB...")

    processed_count = 0
    skipped_count = 0

    for root, _, files in os.walk(input_root_dir):
        for file in files:
            if file.lower().endswith(('.flac', '.wav', '.mp3')):
                input_filepath = os.path.join(root, file)
                relative_path = os.path.relpath(input_filepath, input_root_dir)
                output_filepath = os.path.join(output_root_dir, relative_path)

                # Create necessary subdirectories in the output path
                os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

                try:
                    waveform, sample_rate = torchaudio.load(input_filepath)
                    noisy_waveform = add_gaussian_noise(waveform, sample_rate, snr_db)
                    torchaudio.save(output_filepath, noisy_waveform, sample_rate)
                    processed_count += 1
                except Exception as e:
                    print(f"Error processing {input_filepath}: {e}")
                    skipped_count += 1
    print(f"Finished processing. Processed {processed_count} files, skipped {skipped_count} files.")


# --- Execute processing for training and testing data ---
# You can change the snr_db value to adjust the noise level

# Process the training data
process_audio_directory(TRAIN_DIR, NOISY_TRAIN_DIR, snr_db=15)

# Process the testing data
process_audio_directory(TEST_DIR, NOISY_TEST_DIR, snr_db=15)

print(f"\nNoisy datasets created in '{NOISY_TRAIN_DIR}' and '{NOISY_TEST_DIR}'.")

Processing audio files from '/content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/train-clean-100' to '/content/Transcribing_Model_PyTorch/data/train-noisy-100' with SNR 15dB...
Finished processing. Processed 28539 files, skipped 0 files.
Processing audio files from '/content/Transcribing_Model_PyTorch/data/test-clean/LibriSpeech/test-clean' to '/content/Transcribing_Model_PyTorch/data/test-noisy' with SNR 15dB...
Finished processing. Processed 2620 files, skipped 0 files.

Noisy datasets created in '/content/Transcribing_Model_PyTorch/data/train-noisy-100' and '/content/Transcribing_Model_PyTorch/data/test-noisy'.


# Getting the path of all the transcript files across both train and test directories

In [8]:
import os

def read_and_display_metadata(directory):
    """
    Reads and displays the content of all .txt files in the given directory and its subdirectories.
    """
    print(f"\n--- Metadata Files in: {directory} ---")
    found_files = False
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith('.txt'):
                filepath = os.path.join(root, file)
                print(f"\nContent of: {filepath}")
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        content = f.read()
                        print(content[:500]) # Display first 500 characters to avoid flooding output
                        if len(content) > 500:
                            print("... (truncated)")
                    found_files = True
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
    if not found_files:
        print("No .txt metadata files found.")

# Process metadata files in the original (clean) training and testing directories
# The noisy directories are just copies of the audio, so we refer to the original structure for metadata.
original_train_dir = os.path.join(base_data_path, "train-clean-100/LibriSpeech")
original_test_dir = os.path.join(base_data_path, "test-clean/LibriSpeech")

read_and_display_metadata(original_train_dir)
read_and_display_metadata(original_test_dir)


--- Metadata Files in: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech ---

Content of: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/CHAPTERS.TXT
; Some pipe(|) separated metadata about the audio chapters included in the corpus.
;
; The meaning of the fields in left-to-right order is as follows:
;
; chapter_id: the ID of the chapter in the LibriVox's database
; reader_id: the ID of the reader in the LibriVox's database
; duration: how many minutes of this chapter are used in the corpus
; subset: the corpus subset to which this chapter is assigned
; project_id: the LibriVox project ID
; book_id: the Project Gutenberg's ID for the book on w
... (truncated)

Content of: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/BOOKS.TXT
1      | United States Declaration of Independence          | 
11     | Alice's Adventures in Wonderland                   | 
12     | Through the Looking-Glass                          | 
13     | T

# Checking the cardinality and unique columns of metadata files

In [11]:
import pandas as pd
import numpy as np
import os
from io import StringIO

def read_libri_metadata_file(filepath: str) -> pd.DataFrame:
    """
    Reads a LibriSpeech metadata .txt file, extracts column names based on file type,
    and parses the data into a pandas DataFrame.
    """

    data_lines = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.startswith(';'): # Skip comment lines
                data_lines.append(line.strip())

    # Remove empty lines from data_lines before processing
    data_lines = [line for line in data_lines if line]

    if not data_lines:
        print(f"Warning: No data found in {filepath} after filtering comments. Returning empty DataFrame.")
        return pd.DataFrame()

    col_names = []
    parsed_data = []

    if "BOOKS.TXT" in filepath: # books.txt: book_id | book_title | author
        col_names = ["book_id", "book_title", "author"]
        for line in data_lines:
            parts = [p.strip() for p in line.split('|')]
            # BOOKS.TXT can have 2 or 3 parts (book_id | book_title | author)
            # Pad with NaN if fewer parts than expected
            while len(parts) < len(col_names):
                parts.append(np.nan)
            # Ensure no extra parts if any were there, the last column (author) can't contain '|' here
            parsed_data.append(parts[:len(col_names)])
        df = pd.DataFrame(parsed_data, columns=col_names)
        df['book_id'] = pd.to_numeric(df['book_id'], errors='coerce')
        return df

    elif "CHAPTERS.TXT" in filepath: # chapters.txt: chapter_id | reader_id | duration | subset | project_id | book_id
        col_names = ["chapter_id", "reader_id", "duration", "subset", "project_id", "book_id"]
        for line in data_lines:
            parts = [p.strip() for p in line.split('|')]
            # CHAPTERS.TXT expects 6 parts
            if len(parts) >= len(col_names):
                parsed_data.append(parts[:len(col_names)])
            else:
                print(f"Warning: Line '{line}' in {filepath} has fewer than {len(col_names)} expected fields. Padding with NaN.")
                # Pad with NaN if line is short
                padded_parts = parts + [np.nan] * (len(col_names) - len(parts))
                parsed_data.append(padded_parts)
        df = pd.DataFrame(parsed_data, columns=col_names)
        # Convert numeric-like columns, 'subset' should remain string/categorical
        df['chapter_id'] = pd.to_numeric(df['chapter_id'], errors='coerce')
        df['reader_id'] = pd.to_numeric(df['reader_id'], errors='coerce')
        df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
        # project_id and book_id can be numeric, but might also be strings, coerce to numeric if possible
        df['project_id'] = pd.to_numeric(df['project_id'], errors='coerce')
        df['book_id'] = pd.to_numeric(df['book_id'], errors='coerce')
        return df

    elif "SPEAKERS.TXT" in filepath: # speakers.txt: reader_id | sex | subset | name
        col_names = ["reader_id", "sex", "subset", "name"]
        for line in data_lines:
            parts = line.split('|')
            # Expecting 4 parts: reader_id, sex, subset, name
            # The 'name' field is the last one and can contain '|', so join remaining parts
            if len(parts) >= 4:
                reader_id = parts[0].strip()
                sex = parts[1].strip()
                subset = parts[2].strip()
                name = '|'.join(parts[3:]).strip() # Join remaining parts for the name field
                parsed_data.append([reader_id, sex, subset, name])
            else:
                print(f"Warning: Line '{line}' in {filepath} has fewer than {len(col_names)} expected fields. Padding with NaN.")
                padded_parts = [p.strip() for p in parts] + [np.nan] * (len(col_names) - len(parts))
                parsed_data.append(padded_parts)

        df = pd.DataFrame(parsed_data, columns=col_names)
        df['reader_id'] = pd.to_numeric(df['reader_id'], errors='coerce')
        return df

    else:
        print(f"Warning: Unknown metadata file type '{os.path.basename(filepath)}'. Skipping parsing.")
        return pd.DataFrame()

def profile_dataset_features(df: pd.DataFrame, df_name: str = "DataFrame", display_top_n_categories: int = 10):
    """
    Scans a dataset to automatically break down columns into
    categorical option spaces or numerical statistical boundaries,
    and provides recommendations for high-cardinality features.
    """
    print(f"\n{'='*10} Profiling: {df_name} {'='*10}")
    print(f"Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns\n")

    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns

    print(f"Found {len(categorical_cols)} Categorical columns and {len(numerical_cols)} Numerical columns.\n")
    print("-" * 50)
    print("CATEGORICAL FEATURE OPTIONS DISCOVERY")
    print("-" * 50)

    high_cardinality_cols = []
    id_like_cols = []

    for col in categorical_cols:
        unique_vals = df[col].dropna().unique()
        num_unique = len(unique_vals)
        missing_count = df[col].isna().sum()

        print(f"\n🔹 Feature: '{col}' | Unique Values Count: {num_unique} | Missing: {missing_count} rows")

        # Check for ID-like columns (all unique values)
        if num_unique == len(df) and len(df) > 0:
            id_like_cols.append(col)
            print(f"  💡 This column has all unique values; it's highly likely an ID column.")

        # Flag high cardinality for general categorical columns
        if num_unique > 30: # Threshold for considering a feature high cardinality
            high_cardinality_cols.append(col)
            print(f"  ⚠️ High Cardinality ({num_unique} unique values)! Showing first {min(5, num_unique)} options sample: {list(unique_vals[:min(5, num_unique)])}...")
        elif num_unique > 0:
            # Print value distributions for manageable cardinality
            value_counts = df[col].value_counts(dropna=False)
            # Only display up to display_top_n_categories
            for i, (val, count) in enumerate(value_counts.items()):
                if i >= display_top_n_categories:
                    print(f"  ... (and {num_unique - display_top_n_categories} more unique values)")
                    break
                pct = (count / len(df)) * 100
                print(f"  - [{val}]: {count} occurrences ({pct:.2f}%) ")
        elif num_unique == 0 and missing_count == len(df):
            print("  (All values are missing or column is empty)")

    print("\n" + "-" * 50)
    print("NUMERICAL FEATURE BOUNDARY DISCOVERY")
    print("-" * 50)

    if len(numerical_cols) > 0:
        numeric_summary = df[numerical_cols].describe().T[['min', 'max', 'mean', 'std']]
        print(numeric_summary.to_string())
    else:
        print("No numerical columns found.")

    print("\n" + "=" * 10 + " ANALYSIS & RECOMMENDATIONS " + "=" * 10)

    if id_like_cols:
        print(f"\nPotentially ID-like columns (all unique values in '{df_name}'): {id_like_cols}")
        print("  Recommendation: These columns typically serve as unique identifiers (e.g., `book_id`, `chapter_id`, `reader_id`).")
        print("  - If their sole purpose is identification, consider removing them from your feature set, as they usually don't contribute directly to predictive power and can lead to overfitting.")
        print("  - If they are needed for joining information, keep them for merging, but drop them before model training.")
        print("  - If you believe there's latent information in the ID itself, consider feature engineering (e.g., extracting digits, patterns), but often it's better to use linked metadata.")
    else:
        print("\nNo explicit ID-like columns (all unique) found in this dataset.")

    if high_cardinality_cols:
        print(f"\nHigh-cardinality categorical columns (more than 30 unique values in '{df_name}'): {high_cardinality_cols}")
        print("  Recommendation for high-cardinality features (if not IDs): ")
        print("  - **Grouping rare categories**: Group categories with very low frequency into an 'Other' category.")
        print("  - **Feature Hashing**: Map categories to a fixed-size vector space, which can be memory-efficient.")
        print("  - **Target Encoding**: Replace categories with the mean of the target variable for that category (use with caution to prevent data leakage, e.g., using cross-validation).")
        print("  - **Embedding Layers**: For neural networks, use an embedding layer to learn dense representations for each category.")
        print("  - **Removing irrelevant high-cardinality features** if they don't provide useful information or are too sparse.")
    else:
        print("\nNo high-cardinality categorical columns (more than 30 unique) found that require specific cardinality reduction techniques.")

# --- Paths to specific metadata files --- (using globally defined variables)
# base_data_path is defined in a previous cell
# original_train_dir and original_test_dir are defined in a previous cell

books_file_train = os.path.join(original_train_dir, "BOOKS.TXT")
chapters_file_train = os.path.join(original_train_dir, "CHAPTERS.TXT")
speakers_file_train = os.path.join(original_train_dir, "SPEAKERS.TXT")

# Assuming test directory also contains metadata, though often it's only in train
books_file_test = os.path.join(original_test_dir, "BOOKS.TXT")
chapters_file_test = os.path.join(original_test_dir, "CHAPTERS.TXT")
speakers_file_test = os.path.join(original_test_dir, "SPEAKERS.TXT")

# --- Load and profile metadata files from training directory ---
print("\n" + "#"*10 + " Profiling LibriSpeech Metadata Files (Training Set) " + "#"*10 + "\n")

if os.path.exists(books_file_train):
    df_books_train = read_libri_metadata_file(books_file_train)
    if not df_books_train.empty:
        profile_dataset_features(df_books_train, df_name="BOOKS.TXT (Training)")
else:
    print(f"Warning: {books_file_train} not found. Skipping.")

if os.path.exists(chapters_file_train):
    df_chapters_train = read_libri_metadata_file(chapters_file_train)
    if not df_chapters_train.empty:
        profile_dataset_features(df_chapters_train, df_name="CHAPTERS.TXT (Training)")
else:
    print(f"Warning: {chapters_file_train} not found. Skipping.")

if os.path.exists(speakers_file_train):
    df_speakers_train = read_libri_metadata_file(speakers_file_train)
    if not df_speakers_train.empty:
        profile_dataset_features(df_speakers_train, df_name="SPEAKERS.TXT (Training)")
else:
    print(f"Warning: {speakers_file_train} not found. Skipping.")

print("\n" + "#"*10 + " Profiling LibriSpeech Metadata Files (Test Set) " + "#"*10 + "\n")

# --- Load and profile metadata files from test directory ---
# Note: Test set metadata might be less extensive or even absent for some datasets.
if os.path.exists(books_file_test):
    df_books_test = read_libri_metadata_file(books_file_test)
    if not df_books_test.empty:
        profile_dataset_features(df_books_test, df_name="BOOKS.TXT (Test)")
else:
    print(f"Warning: {books_file_test} not found. Skipping.")

if os.path.exists(chapters_file_test):
    df_chapters_test = read_libri_metadata_file(chapters_file_test)
    if not df_chapters_test.empty:
        profile_dataset_features(df_chapters_test, df_name="CHAPTERS.TXT (Test)")
else:
    print(f"Warning: {chapters_file_test} not found. Skipping.")

if os.path.exists(speakers_file_test):
    df_speakers_test = read_libri_metadata_file(speakers_file_test)
    if not df_speakers_test.empty:
        profile_dataset_features(df_speakers_test, df_name="SPEAKERS.TXT (Test)")
else:
    print(f"Warning: {speakers_file_test} not found. Skipping.")


########## Profiling LibriSpeech Metadata Files (Training Set) ##########


========== Profiling: BOOKS.TXT (Training) ==========
Dataset Shape: 1568 rows | 3 columns

Found 2 Categorical columns and 1 Numerical columns.

--------------------------------------------------
CATEGORICAL FEATURE OPTIONS DISCOVERY
--------------------------------------------------

🔹 Feature: 'book_title' | Unique Values Count: 1506 | Missing: 3 rows
  ⚠️ High Cardinality (1506 unique values)! Showing first 5 options sample: ['United States Declaration of Independence', "Alice's Adventures in Wonderland", 'Through the Looking-Glass', 'The Hunting of the Snark', 'Peter Pan']...

🔹 Feature: 'author' | Unique Values Count: 296 | Missing: 177 rows
  ⚠️ High Cardinality (296 unique values)! Showing first 5 options sample: ['', 'Church of Jesus Christ of Latter-day Saints, Joseph, Jr. Smith, Joseph Smith', 'Frederick Douglass, Frederick Augustus Washington Bailey, Frederick Augustus Washington Baly', 'Mary Wolls

### Deleting Irrelevant Metadata `.txt` Files from Disk

In [17]:
import os

# List of metadata files to delete
files_to_delete = ['BOOKS.TXT', 'CHAPTERS.TXT', 'SPEAKERS.TXT']

# Directories where these files might reside
directories_to_clean = [
    original_train_dir, # defined in cell a3d38f77
    original_test_dir   # defined in cell a3d38f77
]

print("Initiating deletion of specified metadata files...")

for directory in directories_to_clean:
    print(f"\nChecking directory: {directory}")
    for filename in files_to_delete:
        filepath = os.path.join(directory, filename)
        if os.path.exists(filepath):
            try:
                os.remove(filepath)
                print(f"  - Successfully deleted: {filepath}")
            except OSError as e:
                print(f"  - Error deleting {filepath}: {e}")
        else:
            print(f"  - File not found (already deleted or never existed): {filepath}")

print("\nDeletion process complete. Metadata `.txt` files have been removed from disk.")

Initiating deletion of specified metadata files...

Checking directory: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech
  - Successfully deleted: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/BOOKS.TXT
  - Successfully deleted: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/CHAPTERS.TXT
  - Successfully deleted: /content/Transcribing_Model_PyTorch/data/train-clean-100/LibriSpeech/SPEAKERS.TXT

Checking directory: /content/Transcribing_Model_PyTorch/data/test-clean/LibriSpeech
  - Successfully deleted: /content/Transcribing_Model_PyTorch/data/test-clean/LibriSpeech/BOOKS.TXT
  - Successfully deleted: /content/Transcribing_Model_PyTorch/data/test-clean/LibriSpeech/CHAPTERS.TXT
  - Successfully deleted: /content/Transcribing_Model_PyTorch/data/test-clean/LibriSpeech/SPEAKERS.TXT

Deletion process complete. Metadata `.txt` files have been removed from disk.


# Sorting the path of train and test directory

In [ ]:
TRAIN_DIR = "/content/Transcribing_Model_PyTorch/data/train-noisy-100"
TEST_DIR = "/content/Transcribing_Model_PyTorch/data/test-noisy"
print("Classes found in train dir:", sorted(os.listdir(TRAIN_DIR)))
print(len(TRAIN_DIR))
print("Classes found in test dir:", sorted(os.listdir(TEST_DIR)))
print(len(TEST_DIR))

Classes found in train dir: ['103', '1034', '1040', '1069', '1081', '1088', '1098', '1116', '118', '1183', '1235', '1246', '125', '1263', '1334', '1355', '1363', '1447', '1455', '150', '1502', '1553', '1578', '1594', '1624', '163', '1723', '1737', '1743', '1841', '1867', '1898', '19', '1926', '196', '1963', '1970', '198', '1992', '200', '2002', '2007', '201', '2092', '211', '2136', '2159', '2182', '2196', '226', '2289', '229', '233', '2384', '2391', '2416', '2436', '248', '250', '2514', '2518', '254', '26', '2691', '27', '2764', '2817', '2836', '2843', '289', '2893', '2910', '2911', '2952', '298', '2989', '302', '307', '311', '3112', '3168', '32', '3214', '322', '3235', '3240', '3242', '3259', '328', '332', '3374', '3436', '3440', '3486', '3526', '3607', '3664', '3699', '3723', '374', '3807', '3830', '3857', '3879', '39', '3947', '3982', '3983', '40', '4014', '4018', '403', '405', '4051', '4088', '412', '4137', '4160', '4195', '4214', '426', '4267', '4297', '4340', '4362', '4397', '440

# Device-Agnostic code to get to know, which hardware are we using

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda
